# ⚡ Netelpro: Fase 3 - Entrenamiento de Teo-SDS (Silicon Dynamic State)
### *Motor no-transformer, memoria O(1), cero KV-Cache*

Entrena el motor **Netelpro SDS** (`netelpro/neuro/dynamic_state.py`) — reemplaza
la atención de Teo v2 por un espacio de estado dinámico (tipo Mamba/SSM) con
compuertas de silicio formales (STE, banda `[-5000, 5000]`, fail-closed real: si
`control_flag=0`, toda la celda colapsa a 0.0, no solo el estado interno).

Config por defecto (Fase 3 del roadmap): 16 capas, `d_model=1024`, `d_state=16`,
`expand=2` (`d_inner=2048`), vocabulario 32.768 — ~300-350M parámetros.

---
### ⚙️ Configuración en Kaggle:
- **Accelerator:** GPU T4 x2 (o P100)
- **Internet:** On (para clonar el repo — el corpus se monta como Dataset, no se
  vuelve a descargar de HuggingFace)
- **Persistence:** Variables / Files on
- **Input:** montá acá tu Kaggle Dataset con el corpus masivo empaquetado
  (`Add Input` -> el dataset que subiste con `massive_corpus_partial.tar.gz`)


## 1. Verificación de Hardware y GPU T4

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()} | Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## 2. Clonar/Actualizar Repositorio Netelpro (privado)
Mismo patrón que el notebook del corpus: `GITHUB_TOKEN` vía Kaggle Secrets
(`Add-ons -> Secrets`) o pegado manual abajo.

In [ ]:
%cd /kaggle/working
import os

GITHUB_TOKEN = ""  # <-- si no usás Kaggle Secrets, pegá tu token acá

if not GITHUB_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = ""

REPO_URL = (
    f"https://{GITHUB_TOKEN}@github.com/jona2428/netelpro.git"
    if GITHUB_TOKEN
    else "https://github.com/jona2428/netelpro.git"
)

if not os.path.exists('netelpro'):
    !git clone {REPO_URL}
else:
    !cd /kaggle/working/netelpro && git pull origin master
%cd /kaggle/working/netelpro

!pip install -q tokenizers numpy


## 3. Montar el Corpus Masivo (Kaggle Dataset)
Montalo con `Add Input`. Kaggle a veces auto-extrae un `.tar.gz` subido al crear
el Dataset (la carpeta con los `shard_*.bin` ya aparece suelta en `/kaggle/input/...`)
y a veces lo deja empaquetado — esta celda detecta cuál de los dos casos es sola.

In [ ]:
import os
from pathlib import Path

os.makedirs('data', exist_ok=True)

# Caso 1: Kaggle ya auto-extrajo el corpus al crear el Dataset — buscamos
# directo una carpeta con shard_*.bin en cualquier lado bajo /kaggle/input.
shard_dirs = [
    d for d in Path('/kaggle/input').rglob('*')
    if d.is_dir() and list(d.glob('shard_*.bin'))
]

if shard_dirs:
    CORPUS_DIR = shard_dirs[0]
    print(f'✅ Corpus ya extraído en el Dataset, uso directo: {CORPUS_DIR}')
else:
    # Caso 2: sigue empaquetado como .tar.gz, hay que extraerlo nosotros.
    tar_candidates = list(Path('/kaggle/input').rglob('*.tar.gz'))
    assert tar_candidates, (
        'No encontré el corpus ni extraído ni como .tar.gz en /kaggle/input — '
        'montá el Dataset con Add Input antes de correr esta celda.'
    )
    corpus_tar = tar_candidates[0]
    print(f'📦 Extrayendo {corpus_tar} ...')
    !tar -xzf {corpus_tar} -C data
    shard_dirs = [d for d in Path('data').iterdir() if d.is_dir() and list(d.glob('shard_*.bin'))]
    assert shard_dirs, 'La extracción no produjo ningún shard_*.bin — revisá el .tar.gz.'
    CORPUS_DIR = shard_dirs[0]

n_shards = len(list(CORPUS_DIR.glob('shard_*.bin')))
print(f'✅ Corpus listo en {CORPUS_DIR} — {n_shards} shard(s) encontrados.')


## 4. (Opcional) Smoke Test Rápido
Verifica en ~30s que el trainer, el checkpointing y el resume funcionan antes
de lanzar el entrenamiento real de horas. Corre en CPU con datos sintéticos,
no toca tu corpus ni tus checkpoints reales.

In [ ]:
!python training/train_teo_sds.py --smoke-test --checkpoint-dir /kaggle/working/smoke_test_ckpt
!rm -rf /kaggle/working/smoke_test_ckpt training/data/teo_sds_corpus


## 5. Entrenamiento de Teo-SDS (300-350M params) con Auto-Resume
Detecta automáticamente `checkpoint.pt` si ya existe y continúa. Si la sesión
de Kaggle se corta por límite de tiempo, volvé a correr esta misma celda —
retoma en el step exacto donde quedó (optimizador, RNG, todo).

In [ ]:
import os
os.makedirs('models/teo_sds', exist_ok=True)

!python training/train_teo_sds.py \
    --data-dir {CORPUS_DIR} \
    --checkpoint-dir models/teo_sds \
    --device cuda \
    --batch-size 8 \
    --context-len 512 \
    --lr 3e-4 \
    --max-steps 100000 \
    --save-interval 500 \
    --log-interval 50


## 6. Prueba de Inferencia O(1) en Vivo (stepping recurrente, cero KV-Cache)

In [ ]:
import torch
from netelpro.neuro.tokenizer_bpe import NetelproBPETokenizer
from training.train_teo_sds import TeoSDSConfig, load_checkpoint
from netelpro.neuro.dynamic_state import NetelproSDSModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = NetelproBPETokenizer.load('data/teo_v2/tokenizer.json')

ckpt_probe = torch.load('models/teo_sds/checkpoint.pt', map_location=device, weights_only=False)
cfg = TeoSDSConfig.from_dict(ckpt_probe['config'])

model = NetelproSDSModel(cfg.to_sds_config()).to(device)
load_checkpoint('models/teo_sds/checkpoint.pt', model=model, device=device)
model.eval()
print(f'✅ Teo-SDS cargado — step {ckpt_probe["step"]}, loss {ckpt_probe["loss"]:.4f}')

test_prompts = [
    '<|user|>\nHola Teo, soy Jona tu creador. ¿Cómo operan tus compuertas en silicio?<|assistant|>\n',
    '<|user|>\nExplícame cómo funciona un Mutex en Rust con un ejemplo.<|thought|>\n',
]

for p in test_prompts:
    print('=' * 60)
    print('PROMPT:', p)
    input_ids = torch.tensor([tok.encode(p)], device=device)
    out = model.generate(input_ids, max_new_tokens=150, temperature=0.8)
    print('RESPUESTA:', tok.decode(out[0].tolist()))


## 7. Empaquetar Checkpoint Final de Teo-SDS

In [ ]:
!tar -czvf /kaggle/working/teo_sds_checkpoint.tar.gz -C models/teo_sds checkpoint.pt
print('🎉 Checkpoint Teo-SDS empaquetado en /kaggle/working/teo_sds_checkpoint.tar.gz')
